# [B2/B3] Do identifier-poor control candidates pass the negative-control screen unscreened?

**Status:** exploratory  
**Research question:** B2 — follow-on contracts never labelled Phase III; B3 — how much  
Phase III work goes unrecorded ([docs/research-questions.md](../../docs/research-questions.md))
**Decision this informs:** whether the census's retained-control set (the 712 vs 1,029  
comparison in
[studies/phase-iii-census/negative-control-outcomes-2026-08-03.md](../../studies/phase-iii-census/negative-control-outcomes-2026-08-03.md))
needs an unscreenable-candidate partition reported beside it before hand-label
validation is attempted.
**Data as of:** February 2026 data cut (`2026-02-06`); screening artifacts frozen under  
revisions `phase-0-r12` (matching) and `phase-0-r14` (outcomes)
**Owner:** notebooks workflow (Wave 2 seed exploration)  

The negative-control screen excludes a SAM candidate from the eligible pool when its
identifiers collide with resolved SBIR identity (exact UEI/DUNS intersection) or when it
exactly collides with an unresolved SBIR source row's quarantine keys (normalized
name+state or address+ZIP, Revision 11). A candidate whose own identifiers are missing
or malformed *cannot collide on the identifier clause* — it can only be caught by the
quarantine-key clause. The descriptive question: how many screened-negative candidates,
and how many of the 1,029 retained matched controls, were screenable only by quarantine
keys (no usable DUNS beyond the UEI, or malformed identifier fields), and did any reach
the retained-control label without any identifier-clause exposure at all?

This is a screening-coverage question, not a causal claim. By construction
(`build_sam_eligibility_table` raises unless every SAM candidate row has a valid UEI),
a fully identifier-free candidate should be impossible — the notebook verifies that
invariant against the artifact rather than assuming it.

## Data contract

- **Population:** all 887,308 SAM candidate identity envelopes from the frozen
  eligibility artifact; the 843,777 eligible screened negatives; and the 1,029 retained
  matched pairs (712 treated firms).
- **Grain:** candidate identity envelope (exact UEI/DUNS/CAGE co-occurrence group) for
  eligibility; firm for covariates and matches.
- **Keys:** `candidate_ueis` / `candidate_duns` tuples on the eligibility table; UEI on
  covariate and match artifacts. Identifier validity is judged ONLY by the shared
  primitives `sbir_etl.utils.identifiers.normalize_uei` / `normalize_duns` — no local
  re-implementation.
- **Inputs:** `data/processed/phase_iii_negative_controls/` artifacts listed in the
  parameters cell, produced by `scripts/data/build_phase_iii_sam_eligibility.py` and
  `scripts/data/build_phase_iii_control_matches.py`. Expected SHA-256 digests are
  pinned in the study audit docs
  ([control-matching-audit-2026-08-03.md](../../studies/phase-iii-census/control-matching-audit-2026-08-03.md):
  final eligibility `c5c0947d…`, exact matched pairs `4c0aa165…`).
- **Exclusions:** nothing is dropped; every candidate lands in exactly one
  identifier-usability partition.
- **Missingness:** a missing DUNS on a candidate is *absence of a screenable handle*,
  not evidence the candidate is or is not an SBIR firm.
- **Outputs:** exploratory only. If a partition needs to enter the census reporting, it
  becomes a prospective amendment to the frozen spec (`specs/phase-iii-census/`), never
  a notebook export.

In [ ]:
from pathlib import Path

import pandas as pd

from sbir_etl.utils.identifiers import normalize_duns, normalize_uei


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPO_ROOT

In [ ]:
AS_OF_DATE = "2026-02-06"  # February data cut behind every frozen artifact
RANDOM_SEED = 20260807
CONTROLS_DIR = REPO_ROOT / "data" / "processed" / "phase_iii_negative_controls"
REQUIRED_INPUTS: dict[str, Path] = {
    "sam_eligibility": CONTROLS_DIR / "phase_iii_sam_eligibility.parquet",
    "control_eligibility": CONTROLS_DIR / "phase_iii_control_eligibility.parquet",
    "control_covariates": CONTROLS_DIR / "phase_iii_control_covariates.parquet",
    "matches": CONTROLS_DIR / "phase_iii_control_matches.parquet",
}
GENERATORS = {
    "sam_eligibility": "scripts/data/build_phase_iii_sam_eligibility.py",
    "control_eligibility": "scripts/data/build_phase_iii_control_matches.py",
    "control_covariates": "scripts/data/build_phase_iii_control_matches.py",
    "matches": "scripts/data/build_phase_iii_control_matches.py",
}

In [ ]:
input_status = pd.DataFrame(
    [
        {"input": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in REQUIRED_INPUTS.items()
    ]
)
input_status

## Exploration

Each section is one sub-question. All partitions are computed from artifacts; nothing
here invents a count. If an artifact is absent, the cell says so and names the
generator.

In [ ]:
def load_parquet(name: str) -> pd.DataFrame:
    path = REQUIRED_INPUTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first (February inputs required)."
        )
        return pd.DataFrame()
    return pd.read_parquet(path)


eligibility = load_parquet("sam_eligibility")
if not eligibility.empty:
    print(f"{len(eligibility):,} candidate envelopes (audit doc expects 887,308)")
    display(eligibility["eligibility_status"].value_counts().rename("envelopes").to_frame())
eligibility.head()

### Sub-question 1 — partition candidates by identifier usability

Partition every candidate envelope by which identifier handles are usable under the
shared normalization primitives:

- `uei_and_duns` — at least one well-formed UEI and one well-formed DUNS;
- `uei_only` — well-formed UEI, no usable DUNS (screenable on the UEI clause only);
- `duns_only` — usable DUNS but no well-formed UEI (should be impossible per the
  builder's invariant);
- `neither` — no usable identifier at all (must be empty; a nonzero count here means
  the invariant did not hold on the materialized artifact).

In [ ]:
def usable(values, normalizer) -> bool:
    if values is None:
        return False
    try:
        items = list(values)
    except TypeError:
        items = [values]
    return any(normalizer(item) is not None for item in items)


if eligibility.empty:
    partition = pd.DataFrame()
else:
    has_uei = eligibility["candidate_ueis"].map(lambda v: usable(v, normalize_uei))
    has_duns = eligibility["candidate_duns"].map(lambda v: usable(v, normalize_duns))
    eligibility = eligibility.assign(
        identifier_partition=pd.Series(
            pd.NA, index=eligibility.index, dtype="object"
        ).mask(has_uei & has_duns, "uei_and_duns")
        .mask(has_uei & ~has_duns, "uei_only")
        .mask(~has_uei & has_duns, "duns_only")
        .mask(~has_uei & ~has_duns, "neither")
    )
    partition = pd.crosstab(
        eligibility["identifier_partition"], eligibility["eligibility_status"], margins=True
    )
    impossible = eligibility["identifier_partition"].isin(["duns_only", "neither"]).sum()
    print(f"builder-invariant violations (duns_only or neither): {int(impossible)} (expect 0)")
partition

### Sub-question 2 — screen exposure of the `uei_only` screened negatives

For eligible screened negatives with no usable DUNS: they were exposed to the UEI
intersection clause and the quarantine-key clause, but never to the DUNS clause. Read
the recorded `exclusion_reasons` structure on excluded candidates to see which clauses
actually fire in practice, then quantify how much of the eligible pool had reduced
clause exposure.

In [ ]:
if eligibility.empty:
    exposure = pd.DataFrame()
else:
    eligible = eligibility[eligibility["eligibility_status"].eq("eligible_screened_negative")]
    exposure = (
        eligible["identifier_partition"].value_counts()
        .rename("eligible_screened_negatives").to_frame()
        .assign(share=lambda t: (t["eligible_screened_negatives"] / len(eligible)).round(4))
    )
    excluded = eligibility[~eligibility["eligibility_status"].eq("eligible_screened_negative")]
    if "exclusion_reasons" in excluded.columns and not excluded.empty:
        reason_counts = (
            excluded["exclusion_reasons"].map(
                lambda v: tuple(v) if v is not None and not isinstance(v, str) else (v,)
            ).explode().value_counts()
        )
        print("Exclusion-reason frequency among excluded candidates (which clauses fire):")
        display(reason_counts.rename("candidates").to_frame())
exposure

### Sub-question 3 — do reduced-exposure candidates reach the retained-control label?

Join the partition onto the 1,029 retained matched pairs. The decision hinges on this
table: if a material share of retained controls is `uei_only`, the 712 vs 1,029
comparison should carry an unscreenable-candidate partition before hand-label
validation, so labelers know which controls were never DUNS-screenable.

In [ ]:
matches = load_parquet("matches")
control_covariates = load_parquet("control_covariates")
if eligibility.empty or matches.empty:
    retained_partition = pd.DataFrame()
elif "identifier_partition" not in eligibility.columns:
    print("Run the identifier-partition cell above before this join.")
    retained_partition = pd.DataFrame()
else:
    print(f"matched pair rows: {len(matches):,} (audit doc expects 1,029)")
    display(matches.head())
    # Identify the control-side UEI column on the matches artifact by inspection —
    # the frozen builder names columns explicitly; adjust here if it differs.
    control_key_candidates = [
        column for column in matches.columns if "control" in column.lower() and "uei" in column.lower()
    ]
    if not control_key_candidates:
        print("No control-UEI column recognized on the matches artifact; "
              f"columns are: {list(matches.columns)}")
        retained_partition = pd.DataFrame()
    else:
        control_key = control_key_candidates[0]
        # Vectorized normalized-UEI -> partition map. Explode the per-envelope UEI tuples
        # to one row per (UEI, partition), normalize once through the shared primitive,
        # drop unusable keys, then keep one partition per normalized UEI. This replaces a
        # nested per-row/per-UEI Python loop and scales to the ~887k-envelope artifact.
        # keep="last" reproduces the loop's last-write-wins; by construction a normalized
        # UEI belongs to exactly one exact envelope, so the choice does not change results.
        exploded = eligibility[["candidate_ueis", "identifier_partition"]].explode("candidate_ueis")
        exploded["norm_uei"] = exploded["candidate_ueis"].map(normalize_uei)
        envelope_partition = (
            exploded.dropna(subset=["norm_uei"])
            .drop_duplicates(subset=["norm_uei"], keep="last")
            .set_index("norm_uei")["identifier_partition"]
        )
        retained = matches[[control_key]].drop_duplicates().copy()
        retained["identifier_partition"] = (
            retained[control_key].map(normalize_uei).map(envelope_partition)
        )
        retained_partition = (
            retained["identifier_partition"].value_counts(dropna=False)
            .rename("retained_control_firms").to_frame()
        )
retained_partition

### Sub-question 4 — quarantine-clause reach for the reduced-exposure controls

The identifier-poor SBIR side (33,062 unresolved source rows; see the
[identity-eligibility audit](../../studies/phase-iii-census/identity-eligibility-audit-2026-08-03.md))
is screened only via exact name+state / address+ZIP collision. For retained controls in
the `uei_only` partition, the follow-up is whether their SAM name/state/address fields
were even populated well enough to collide — an empty address field on the candidate
side silently narrows the quarantine clause the same way a missing DUNS narrows the
identifier clause. Scaffold: pull the quarantine-key fields for the retained `uei_only`
controls from the eligibility table (or the SAM entity extract, if the fields are not
carried through) and tabulate field completeness. Left as the next work item — the
column layout of the materialized artifact decides where those fields are read from.

## Findings and caveats

| Claim | Evidence/artifact | Caveat or alternative explanation |
|---|---|---|
| _Draft — no counts asserted until the artifacts are present_ | _Partition/crosstab cells above_ | _A `uei_only` control may still be a perfectly clean negative; reduced exposure is not misclassification_ |

Separate observations from interpretation. Any partition that should accompany the
712 vs 1,029 comparison enters through a prospective amendment to the frozen census
spec — a number computed here is not citable and does not update the study.

## Graduation checklist

- [ ] Question and decision are explicit.
- [ ] Data snapshot, population, grain, keys, and exclusions are recorded.
- [ ] Samples and stochastic methods use a deterministic seed.
- [ ] Reusable calculations have typed functions and unit tests.
- [ ] Recurring artifact generation has a thin CLI or Dagster asset.
- [ ] Published figures have an independent verifier.
- [ ] Findings and methodology are linked from `docs/`.
- [ ] Outputs and execution counts are cleared before commit.